In [1]:
import os
import sys
import jax
import flax
import optax
from jax import lax, random, value_and_grad, numpy as jnp
from jax import random, grad, vmap, hessian, jacfwd, jit
from jax import config
from jax.tree_util import tree_map

import torch
from torch.utils.data import Dataset
import jax_dataloader as jd
from pdetransformer.core.mixed_channels import PDETransformer

import time
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt

os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false' 
config.update("jax_enable_x64", True)

print(f"JAX backend: {jax.default_backend()}")
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")

/home/tanki/projects/jax_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


JAX backend: gpu
PyTorch CUDA available: True


In [2]:
beta = 10.0
DATA_DIR = Path.cwd().parent.parent / "data" / "darcy_flow_data" 
FILE_PATH = DATA_DIR / "2D_DarcyFlow_beta10.0_Train_Scale1.hdf5"

print(f"Loading pre-processed dataset: {FILE_PATH.name}")

with h5py.File(FILE_PATH, 'r') as f:
    fine_x_coords = np.array(f['x-coordinate'][:])
    fine_y_coords = np.array(f['y-coordinate'][:])
    
    all_a_flat = np.array(f['a_flat'][:])
    all_a_x_flat = np.array(f['a_x_flat'][:])
    all_a_y_flat = np.array(f['a_y_flat'][:])
    all_u_flat = np.array(f['u_flat'][:])

N_samples = all_a_flat.shape[0]
fine_nx, fine_ny = len(fine_x_coords), len(fine_y_coords)

# Coordinate generation
X, Y = np.meshgrid(fine_x_coords, fine_y_coords, indexing='ij')
x = jnp.array(X.reshape(-1, 1))
y = jnp.array(Y.reshape(-1, 1))

# Generate Mathematical Boundaries
edge_x = jnp.linspace(0.0, 1.0, fine_nx)
edge_y = jnp.linspace(0.0, 1.0, fine_ny)

bc_bottom = jnp.stack([edge_x, jnp.zeros_like(edge_x)], axis=-1)
bc_top = jnp.stack([edge_x, jnp.ones_like(edge_x)], axis=-1)
bc_left = jnp.stack([jnp.zeros_like(edge_y), edge_y], axis=-1)
bc_right = jnp.stack([jnp.ones_like(edge_y), edge_y], axis=-1)

all_bc_coords = jnp.concatenate([bc_bottom, bc_top, bc_left, bc_right], axis=0)
x_bc = all_bc_coords[:, 0].reshape(-1, 1)
y_bc = all_bc_coords[:, 1].reshape(-1, 1)
bc_val_array = jnp.zeros_like(x_bc)

print(f"Grid Size: {fine_nx}x{fine_ny} ({x.shape[0]} total points)")

Loading pre-processed dataset: 2D_DarcyFlow_beta10.0_Train_Scale1.hdf5
Grid Size: 128x128 (16384 total points)


In [3]:
from torch.utils.data import DataLoader

class DarcyFlowDataset(Dataset):
    def __init__(self, a_data, a_x_data, a_y_data, u_data):
        self.a = a_data
        self.a_x = a_x_data
        self.a_y = a_y_data
        self.u = u_data

    def __len__(self): return len(self.a)
    def __getitem__(self, idx): return self.a[idx], self.a_x[idx], self.a_y[idx], self.u[idx]

n_train, n_val, n_test = 32, 10, 10
N_subset = n_train + n_val + n_test

train_dataset = DarcyFlowDataset(all_a_flat[:n_train], all_a_x_flat[:n_train], all_a_y_flat[:n_train], all_u_flat[:n_train])
val_dataset = DarcyFlowDataset(all_a_flat[n_train : n_train + n_val], all_a_x_flat[n_train : n_train + n_val], all_a_y_flat[n_train : n_train + n_val], all_u_flat[n_train : n_train + n_val])
test_dataset = DarcyFlowDataset(all_a_flat[n_train + n_val : N_subset], all_a_x_flat[n_train + n_val : N_subset], all_a_y_flat[n_train + n_val : N_subset], all_u_flat[n_train + n_val : N_subset])

batch_size_samples = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size_samples, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=n_val, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=n_test, shuffle=False)

print(f"DataLoaders ready! (Train: {n_train}, Val: {n_val}, Test: {n_test})")

DataLoaders ready! (Train: 32, Val: 10, Test: 10)


In [4]:
print("Loading PDE-Transformer from HuggingFace...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Using the small mixed-channel pre-trained variant
fm_model = PDETransformer.from_pretrained('thuerey-group/pde-transformer', subfolder='mc-s').to(device)
fm_model.eval()

print("\n--- Model Architecture (Last 15 Modules) ---")
module_list = list(fm_model.named_modules())
for name, module in module_list[-15:]:
    print(f"{name}: {module.__class__.__name__}")

Loading PDE-Transformer from HuggingFace...


/home/tanki/projects/jax_env/lib/python3.12/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



--- Model Architecture (Last 15 Modules) ---
model.decoder_level_1.blocks.1.mlp.fc1: Linear
model.decoder_level_1.blocks.1.mlp.act: GELU
model.decoder_level_1.blocks.1.mlp.fc2: Linear
model.decoder_level_1.blocks.1.mlp.drop: Dropout
model.decoder_level_1.blocks.1.adain_2: AdaLayerNormZero
model.decoder_level_1.blocks.1.adain_2.silu: SiLU
model.decoder_level_1.blocks.1.adain_2.linear: Linear
model.decoder_level_1.blocks.1.adain_2.norm: LayerNorm
model.output: Conv2d
model.final_layer: FinalLayer
model.final_layer.norm_final: LayerNorm2d
model.final_layer.out_proj: Conv2d
model.final_layer.adaLN_modulation: Sequential
model.final_layer.adaLN_modulation.0: SiLU
model.final_layer.adaLN_modulation.1: Linear


In [7]:
# Dictionary to catch the intercepted features
latent_features = {}

def get_features_hook(module, args):
    latent_features['phi'] = args[0].detach()

# 1. Attach the hook directly before the final Conv2d projection
hook_handle = fm_model.model.final_layer.out_proj.register_forward_pre_hook(get_features_hook)

# 2. Grab a single test sample from your dataset
a_sample, a_x_sample, a_y_sample, u_sim_sample = test_dataset[0]

# 3. Format the Darcy field for the PDE-Transformer (2 Channels)
# Channel 0: a(x,y)
a_tensor = torch.tensor(a_sample.reshape(fine_nx, fine_ny), dtype=torch.float32)
a_tensor = a_tensor.unsqueeze(0).unsqueeze(0).to(device)

# Channel 1: f(x,y) = beta = 10.0
beta_val = 10.0
f_tensor = torch.full_like(a_tensor, beta_val)

# Stack them along the channel dimension (dim=1)
input_tensor = torch.cat([a_tensor, f_tensor], dim=1)

print(f"Input tensor shape: {input_tensor.shape}")

# 4. Push the 2-channel data through the foundation model
with torch.no_grad():
    try:
        _ = fm_model(input_tensor)
        print("Forward pass successful.")
    except Exception as e:
        print(f"Forward pass failed: {e}")

# 5. Extract, Reshape, and Bridge to JAX
if 'phi' in latent_features:
    extracted_tensor = latent_features['phi']
    print(f"Intercepted PyTorch latent shape: {extracted_tensor.shape}")
    
    # Squeeze batch, permute to (Height, Width, Hidden_Dim), and flatten spatial dimensions
    features_np = extracted_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    features_flat = features_np.reshape(-1, features_np.shape[-1])
    
    # Bridge to JAX
    f_eval_jax = jnp.array(features_flat)
    
    print(f"Final JAX feature matrix (Phi) shape: {f_eval_jax.shape}")
    
# Clean up the hook so it doesn't leak memory on future passes
hook_handle.remove()

Input tensor shape: torch.Size([1, 2, 128, 128])
Forward pass successful.
Intercepted PyTorch latent shape: torch.Size([1, 192, 32, 32])
Final JAX feature matrix (Phi) shape: (1024, 192)
